In [1]:
# BLOCK 0  – GLOBAL SETUP: imports ▸ paths ▸ flags ▸ constants
# ════════════════════════════════════════════════════════════════════════════════

# --- core libs
import numpy as np
import nibabel as nib
import pandas as pd
import gc                            # ← added
import torch                         # ← added

# --- imaging / DL
import cv2, matplotlib.pyplot as plt
from ultralytics import YOLO
from totalsegmentator.python_api import totalsegmentator

# --- misc libs
from pathlib import Path
from collections import Counter

# ── PATHS (EDIT HERE) ───────────────────────────────────────────────────────────
MODEL_PATH = Path(r"C:\Users\Ryan Krishna\Documents\Overscanning\yolo_runs\yolo11_pubic_symphysis_m_hardtrain\weights\best.pt")

NIFTI_DIR  = Path(r"D:\Abdomen_CT_Bone_Mets_Nifti")
CSV_PATH   = NIFTI_DIR / "overscanning_results.csv"

# ── FLAGS ───────────────────────────────────────────────────────────────────────
DISPLAY_DETECTION = True     # draw green box on best slice
FAST_MODEL        = False    # TotalSegmentator "fast" mode
MULTI_LABEL_MASK  = True     # 1 = liver, 2 = spleen

# ── CONSTANTS ───────────────────────────────────────────────────────────────────
FINAL_CONF    = 0.20     # YOLO confidence threshold
BACKGROUND_HU = -300     # HU ≤ –300 → treated as air/outside body
model         = YOLO(str(MODEL_PATH))   # load once, reused everywhere

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
#  Pubic‑symphysis detection ➔ caudal‑overscan  (crash‑resistant, incremental CSV)
# ════════════════════════════════════════════════════════════════════════════════
import random, traceback, pandas as pd, numpy as np, nibabel as nib, cv2, matplotlib.pyplot as plt
from pathlib import Path

# ────────────────────────────────────────────────────────────────────────────────
# 0)  LOAD ALREADY‑PROCESSED FILE LIST (if any)
# ────────────────────────────────────────────────────────────────────────────────
if CSV_PATH.exists():
    done_df = pd.read_csv(CSV_PATH)
    done_set = set(done_df["file_name"].tolist())
    print(f"↪️  {len(done_set)} scans already in CSV – they’ll be skipped\n")
else:
    done_set = set()

# ────────────────────────────────────────────────────────────────────────────────
# 1)  ORIGINAL BLOCK‑1 HELPERS  (UNCHANGED)
# ────────────────────────────────────────────────────────────────────────────────
def preprocess_slice(ct_slice: np.ndarray) -> np.ndarray:
    arr = ct_slice.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    arr = (arr * 255.0).astype(np.uint8)
    return cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)

def find_best_symphysis_slice(ct_path: Path, show: bool = True) -> int | None:
    ct      = nib.load(str(ct_path))
    H, W, Z = ct.shape
    vol     = ct.get_fdata()

    best_conf, best_slice, best_box = -1.0, None, None
    for z in range(Z):
        img = preprocess_slice(vol[:, :, z])
        res = model.predict(img, conf=FINAL_CONF, device=0, save=False)[0]

        for b in sorted(res.boxes, key=lambda bb: float(bb.conf), reverse=True):
            conf           = float(b.conf)
            x1, y1, x2, y2 = b.xyxy[0].tolist()

            # 1) outside‑body / air test
            centre_hu = float(vol[int((y1 + y2)/2), int((x1 + x2)/2), z])
            if centre_hu <= BACKGROUND_HU:
                continue
            # 2) mid‑line test (±20 % width)
            cx      = int((x1 + x2)/2)
            if abs(cx - W//2) > (W * 0.20):
                continue
            # 3) bone HU test (mean ≥150 HU in 20×20 window)
            pad = 10
            x0, x1w = max(0, cx-pad),                min(W, cx+pad)
            y0, y1w = max(0, int((y1+y2)/2)-pad),    min(H, int((y1+y2)/2)+pad)
            if vol[y0:y1w, x0:x1w, z].mean() < 150:
                continue
            # 4) take highest‑conf valid box
            if conf > best_conf:
                best_conf, best_slice, best_box = conf, z, (x1, y1, x2, y2)
            break

    if best_slice is None:
        print(f"❌ {ct_path.name}: no valid detection")
        return None

    if show:
        x1, y1, x2, y2 = best_box
        vis = preprocess_slice(vol[:, :, best_slice]).copy()
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), (0,255,0), 2)
        plt.figure(figsize=(5,5)); plt.axis("off")
        plt.title(f"{ct_path.name} – slice {best_slice} (conf={best_conf:.3f})")
        plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.show()

    return best_slice

# ────────────────────────────────────────────────────────────────────────────────
# 2)  LOOP THROUGH ALL NIFTIs, APPENDING TO CSV ONE‑BY‑ONE
# ────────────────────────────────────────────────────────────────────────────────
nii_paths = sorted(NIFTI_DIR.rglob("*.nii*"))
print(f"🔎 Found {len(nii_paths)} NIfTI files in total\n")

for ct_path in nii_paths:
    if ct_path.name in done_set:
        continue   # already completed earlier

    try:
        print(f"▶ {ct_path.relative_to(NIFTI_DIR.parent)}")
        z = find_best_symphysis_slice(ct_path, show=False)
        if z is None:
            continue  # detection failed, nothing to save

        # ── ORIGINAL BLOCK‑2 CALCULATION  (UNCHANGED)───────────────────────────
        ct_obj  = nib.load(str(ct_path))
        affine  = ct_obj.affine
        Z       = ct_obj.shape[2]

        pubic_z = float((affine @ np.array([0,0, z,1]))[2])
        end_z   = min(float((affine @ np.array([0,0,k,1]))[2]) for k in range(Z))
        caudal_mm = abs(end_z - pubic_z)

        row = {
            "file_name":          ct_path.name,
            "pubic_z_mm":         int(round(pubic_z)),
            "scan_end_z_mm":      int(round(end_z)),
            "caudal_overscan_mm": int(round(caudal_mm)),
        }

        # ────────────────────────────────────────────────────────────────────
        #  APPEND / UPDATE CSV  *IMMEDIATELY*
        # ────────────────────────────────────────────────────────────────────
        new_df = pd.DataFrame([row])
        if CSV_PATH.exists():
            # append & deduplicate on file_name (keep latest)
            tmp = pd.read_csv(CSV_PATH)
            tmp = pd.concat([tmp, new_df], ignore_index=True)
            tmp = tmp.drop_duplicates(subset="file_name", keep="last")
            tmp.sort_values("file_name").to_csv(CSV_PATH, index=False)
        else:
            new_df.to_csv(CSV_PATH, index=False)

        done_set.add(ct_path.name)         # mark as done for this run
        print("   ✓ saved")

    except Exception as e:
        print("   ⚠️  error – continuing")
        traceback.print_exc(limit=1)

print(f"\n✅ Finished. Total rows now in CSV: {len(done_set)}")


In [2]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 3  – TotalSegmentator (liver + spleen) → combined masks
# ════════════════════════════════════════════════════════════════════════════════
def ensure_liver_spleen_mask(ct_path: Path) -> Path:
    out_dir      = ct_path.parent / "ts_liver_spleen"
    liver_mask   = out_dir / "liver.nii.gz"
    spleen_mask  = out_dir / "spleen.nii.gz"
    merged_mask  = ct_path.parent / "liver_spleen_combined.nii.gz"

    if merged_mask.exists():
        return merged_mask

    # 1) run TotalSegmentator if needed
    if not (liver_mask.exists() and spleen_mask.exists()):
        out_dir.mkdir(exist_ok=True)
        totalsegmentator(
            ct_path, out_dir,
            roi_subset=["liver", "spleen"],
            task="total",
            fast=FAST_MODEL,
            device="gpu"
        )

    # 2) merge masks
    liver_img   = nib.load(liver_mask)
    spleen_img  = nib.load(spleen_mask)
    liver_data  = liver_img.get_fdata() > 0
    spleen_data = spleen_img.get_fdata() > 0

    if MULTI_LABEL_MASK:
        combined = np.zeros(liver_data.shape, dtype=np.uint8)
        combined[liver_data]  = 1
        combined[spleen_data] = 2
    else:
        combined = (liver_data | spleen_data).astype(np.uint8)

    merged_img = nib.Nifti1Image(combined, liver_img.affine, liver_img.header)
    nib.save(merged_img, merged_mask)

    # 3) delete individual masks
    for f in (liver_mask, spleen_mask):
        if f.exists():
            f.unlink()

    return merged_mask

nii_paths = sorted(NIFTI_DIR.rglob("*.nii*"))
print(f"Found {len(nii_paths)} NIfTI files\n")

# --- run segmentation (skip if combined already present) ---
for ct_path in nii_paths:
    rel = ct_path.relative_to(NIFTI_DIR.parent)
    print(f"▶ Segmentation check: {rel}")
    ensure_liver_spleen_mask(ct_path)

print("\n✓ Liver + spleen masks ensured for all scans")

Found 1057 NIfTI files

▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000001\BMAB1_00000001.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000001\liver_spleen_combined.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000002\BMAB1_00000002.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000002\liver_spleen_combined.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000003\BMAB1_00000003.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000003\liver_spleen_combined.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000004\BMAB1_00000004.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000004\liver_spleen_combined.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000005\BMAB1_00000005.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000005\liver_spleen_combined.nii.gz
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB1_00000011\BMAB1_00000011.ni

100%|██████████| 8/8 [00:00<00:00, 28.65it/s]


  Predicted in 6.54s
Resampling...
  cropping from (512, 512, 1018) to (330, 248, 382)
Resampling...
  Resampled in 1.10s
Predicting part 1 of 1 ...


100%|██████████| 4/4 [00:00<00:00,  5.09it/s]


  Predicted in 7.39s
Resampling...
Saving segmentations...
Shape of output image is very big. Setting nr_threads_saving=1 to save memory.
  Saved in 3.21s
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB3_00000067\BMAB3_00000067.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 5.56s
Predicting...


...

  Predicted in 6.93s
Resampling...
Saving segmentations...
  Saved in 4.62s
▶ Segmentation check: Abdomen_CT_Bone_Mets_Nifti\BMAB3_00000250\BMAB3_00000250.nii.gz

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 1.07s
Predicting...


100%|██████████| 4/4 [00:00<00:00, 216.05it/s]


  Predicted in 6.25s
Resampling...
  cropping from (751, 590, 97) to (580, 437, 43)
Resampling...
  Resampled in 1.15s
Predicting part 1 of 1 ...


100%|██████████| 12/12 [00:00<00:00, 28.28it/s]


  Predicted in 7.44s
Resampling...
Saving segmentations...
  Saved in 5.22s

✓ Liver + spleen masks ensured for all scans


In [3]:
# ════════════════════════════════════════════════════════════════════════════════
# BLOCK 4  – Cranial overscan  + append to existing overscanning_results.csv
#            (adds four cranial columns, including which organ set the slice)
# ════════════════════════════════════════════════════════════════════════════════
def cranial_overscan(ct_path: Path, mask_path: Path) -> tuple[int, int, int, str]:
    """
    Returns
        cranial_mm      – rounded overscan distance
        organ_z_mm      – world‑z of cranial‑most liver/spleen slice
        scan_start_mm   – world‑z of cranial scan edge used
        organ_top       – "Liver" or "Spleen" that defined the cranial slice
    """
    ct_img   = nib.load(str(ct_path))
    mask_img = nib.load(str(mask_path))
    affine   = ct_img.affine
    Z        = ct_img.shape[2]
    mask_np  = mask_img.get_fdata()

    seg_slices = np.where(mask_np.any(axis=(0, 1)))[0]
    if seg_slices.size == 0:
        raise RuntimeError(f"{ct_path.name}: empty combined mask")

    # world‑space z for each liver/spleen slice
    z_coords = [(k, float((affine @ [0, 0, k, 1])[2])) for k in seg_slices]

    # world‑space z of both scan edges
    z_edge0 = float((affine @ [0, 0,      0, 1])[2])
    z_edgeN = float((affine @ [0, 0, Z - 1, 1])[2])

    cranial_edge_z = z_edge0 if z_edge0 > z_edgeN else z_edgeN
    highest_slice, highest_z = min(z_coords, key=lambda t: abs(t[1] - cranial_edge_z))

    # which organ owns the cranial slice?
    labels     = mask_np[:, :, highest_slice][mask_np[:, :, highest_slice] > 0].astype(int)
    organ_map  = {1: "Liver", 2: "Spleen"}
    organ_top  = organ_map.get(int(np.bincount(labels).argmax()), "Unknown")

    cranial_mm    = int(round(abs(cranial_edge_z - highest_z)))
    scan_start_mm = int(round(cranial_edge_z))
    organ_z_mm    = int(round(highest_z))
    return cranial_mm, organ_z_mm, scan_start_mm, organ_top


# ── build cranial DataFrame ─────────────────────────────────────────────────────
rows_cranial = []
for ct_path in nii_paths:
    mask_path = ct_path.parent / "liver_spleen_combined.nii.gz"
    if not mask_path.exists():
        print(f"⚠️  {mask_path.name} missing → skipping {ct_path.name}")
        continue
    cranial_mm, organ_z_mm, scan_start_mm, organ_top = cranial_overscan(ct_path, mask_path)
    rows_cranial.append({
        "file_name":           ct_path.name,
        "liver_spleen_z_mm":   organ_z_mm,
        "scan_start_z_mm":     scan_start_mm,
        "cranial_overscan_mm": cranial_mm,
        "top_organ":           organ_top,
    })

df_cranial = pd.DataFrame(rows_cranial)

# ── load existing CSV and append/overwrite cranial columns ──────────────────────
existing = pd.read_csv(CSV_PATH) if CSV_PATH.exists() else pd.DataFrame()
df_final = existing.merge(df_cranial, on="file_name", how="inner", suffixes=("", "_new"))

# resolve any duplicate columns created by the merge
for col in ["liver_spleen_z_mm", "scan_start_z_mm", "cranial_overscan_mm", "top_organ"]:
    if f"{col}_new" in df_final.columns:
        df_final[col] = df_final[f"{col}_new"].fillna(df_final[col])
        df_final.drop(columns=f"{col}_new", inplace=True)

# cast numeric columns to plain int (safe now – no NaN after inner merge)
num_cols = df_final.select_dtypes(include="number").columns
df_final[num_cols] = df_final[num_cols].astype(int)

df_final.sort_values("file_name", inplace=True)
df_final.to_csv(CSV_PATH, index=False)

print(f"\n✅ Cranial metrics appended to existing CSV → {CSV_PATH.resolve()}")
df_final.head()


✅ Cranial metrics appended to existing CSV → D:\Abdomen_CT_Bone_Mets_Nifti\overscanning_results.csv


,file_name,pubic_z_mm,scan_end_z_mm,caudal_overscan_mm,liver_spleen_z_mm,scan_start_z_mm,cranial_overscan_mm,top_organ
0,BMAB1_00000001.nii.gz,1274,1216,58,1661,1673,12,Liver
1,BMAB1_00000002.nii.gz,-1027,-1085,58,-621,-593,28,Liver
2,BMAB1_00000003.nii.gz,1251,1180,71,1623,1640,18,Liver
3,BMAB1_00000004.nii.gz,-655,-726,71,-275,-249,26,Liver
4,BMAB1_00000005.nii.gz,-474,-585,110,-74,-42,32,Liver


In [ ]:
# # ════════════════════════════════════════════════════════════════════════════════
# # Replace "file_name" with patient ID and delete the extra column
# # ════════════════════════════════════════════════════════════════════════════════
# import pandas as pd

# CSV_PATH = r"D:\Abdomen_CT_Bone_Mets_Nifti\overscanning_results.csv"  # adjust if needed
# DRY_RUN  = False  # ▶ False → write changes   |   True → preview only

# # ── load CSV ────────────────────────────────────────────────────────────────────
# df = pd.read_csv(CSV_PATH)

# # derive ID (first two underscore‑separated parts) and overwrite file_name
# new_ids = (
#     df["file_name"]
#       .str.replace(".nii.gz", "", regex=False)
#       .str.extract(r"^([^_]+_[0-9]+)", expand=False)
# )

# df["file_name"] = new_ids

# # drop the old helper column if it exists
# df.drop(columns=[c for c in ("patient_id",) if c in df.columns], inplace=True)

# # ── preview or save ─────────────────────────────────────────────────────────────
# if DRY_RUN:
#     print("\n🟡 DRY‑RUN: preview after replacement (first 8 rows)\n")
#     display(df.head(8))
# else:
#     df.to_csv(CSV_PATH, index=False)
#     print(f"✅ CSV updated – patient IDs now in 'file_name' at → {CSV_PATH}")